In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # 1 = Quadro P6000 in listing

import torch
import pandas as pd, numpy as np
import transformers, timm
from transformers import pipeline, AutoImageProcessor
from transformers.image_utils import load_image
from tqdm import tqdm

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of visible GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}  ->  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ------------------ pooling helper ------------------
def _pool_feature(feat):
    """
    Convert one pipeline output to a 1D vector.
    Handles shapes: (D,), (1,D), (T,D), (1,T,D), (B,T,D). Returns mean over tokens -> (D,).
    """
    arr = np.asarray(feat)
    if arr.ndim == 1:         # (D,)
        return arr
    if arr.ndim == 2:         # (1,D) or (T,D)
        return arr.mean(axis=0)
    if arr.ndim == 3:         # (1,T,D) or (B,T,D)
        # average everything except the last dim (feature dim)
        return arr.mean(axis=tuple(range(arr.ndim - 1)))
    raise ValueError(f"Unexpected feature shape: {arr.shape}")

# ------------------ batched extraction ------------------
def extract_embeddings_batched(csv_path: str, emb_out_path: str, lbl_out_path: str,
                               batch_size: int = 64, seed: int = 42):
    """
    Reads a manifest CSV with ['path','label'], extracts RAD-DINO embeddings in batches on GPU,
    pools token outputs to a single vector/image, and saves arrays to disk.
    """
    df = pd.read_csv(csv_path)
    if not {"path", "label"}.issubset(df.columns):
        raise ValueError(f"{csv_path} must have columns ['path','label']")

    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    paths  = df["path"].tolist()
    labels = df["label"].to_numpy(dtype=np.int64)

    # Dry-run for pooled vector dimension
    vec0 = _pool_feature(feature_extractor(load_image(paths[0]))[0])
    dim  = int(vec0.shape[0])
    print(f"Detected embedding dim after pooling: {dim}")

    # Preallocate
    embs = np.empty((len(paths), dim), dtype=np.float32)

    idx = 0
    torch.set_grad_enabled(False)
    with torch.inference_mode():
        for start in tqdm(range(0, len(paths), batch_size), desc=f"Extracting ({os.path.basename(csv_path)})"):
            end = min(start + batch_size, len(paths))
            images = [load_image(p) for p in paths[start:end]]

            feats  = feature_extractor(images)                       # list len = batch size
            batch  = np.stack([_pool_feature(f) for f in feats])     # (B, D)
            bsz    = end - start

            embs[idx:idx+bsz] = batch.astype(np.float32, copy=False)
            idx += bsz

    np.save(emb_out_path, embs)
    np.save(lbl_out_path, labels)
    print(f"Saved {emb_out_path} {embs.shape}  |  {lbl_out_path} {labels.shape}")
    return embs, labels

In [ ]:
# ------------------ config ------------------
MODEL_NAME = "microsoft/rad-dino"
HF_TOKEN   = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # your token

TRAIN_CSV = "/home/jupyter-yin10/Image_Analysis/random_s8/train_manifest.csv"
TEST_CSV  = "/home/jupyter-yin10/Image_Analysis/random_s8/test_manifest.csv"

TRAIN_EMB_OUT = "/home/jupyter-yin10/Image_Analysis/random_s8/train_emb_rad_dino.npy"
TRAIN_LBL_OUT = "/home/jupyter-yin10/Image_Analysis/random_s8/train_labels_rad_dino.npy"
TEST_EMB_OUT  = "/home/jupyter-yin10/Image_Analysis/random_s8/test_emb_rad_dino.npy"
TEST_LBL_OUT  = "/home/jupyter-yin10/Image_Analysis/random_s8/test_labels_rad_dino.npy"

BATCH_SIZE = 64   # if VRAM is tight on P6000, try 32 or 16
SEED = 42
np.random.seed(SEED)

In [ ]:
# ------------------ pipeline ------------------
# Use fast processor to avoid the warning; P6000 runs fine in fp32
proc = AutoImageProcessor.from_pretrained(MODEL_NAME, use_fast=True)
feature_extractor = pipeline(
    task="image-feature-extraction",
    model=MODEL_NAME,
    device=0,                                    # cuda:0 (P6000 due to CUDA_VISIBLE_DEVICES)
    token=HF_TOKEN,
    image_processor=proc,
    model_kwargs={"torch_dtype": torch.float32}   # fp32 preferred on P6000
)


In [ ]:
# ------------------ run: TRAIN & TEST ------------------
train_embs, train_labels = extract_embeddings_batched(
    TRAIN_CSV, TRAIN_EMB_OUT, TRAIN_LBL_OUT, batch_size=BATCH_SIZE, seed=SEED
)
test_embs,  test_labels  = extract_embeddings_batched(
    TEST_CSV,  TEST_EMB_OUT,  TEST_LBL_OUT,  batch_size=BATCH_SIZE, seed=SEED
)

print("Train:", train_embs.shape, train_labels.shape)
print("Test: ", test_embs.shape,  test_labels.shape)